In [0]:
from pyspark.sql import functions as F

# Read the  taxi table and print basic info: row count, column count, schema, and summary stats
df = spark.read.table("group3_gp.testing.for_hire_vehicles")
print(f"Row count : {df.count():,}")
print(f"Columns   : {len(df.columns)}")
df.printSchema()


In [0]:
%sql
-- ════════════════════════════════════════════════
-- FOR HIRE VEHICLES — Bad data checks
-- Only 8 columns — no fare or distance data
-- dropoff only available from 2017 onwards
-- ════════════════════════════════════════════════

-- Total row count
SELECT 'Total rows'                    AS check_name, COUNT(*) AS result
FROM group3_gp.testing.for_hire_vehicles

UNION ALL

-- Trips where dropoff is before or equal to pickup — likely GPS or logging errors
SELECT 'Bad timestamps (dropoff <= pickup)' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.for_hire_vehicles
WHERE TO_TIMESTAMP(dropoff_datetime) <= TO_TIMESTAMP(pickup_datetime)

UNION ALL

-- Dirty affiliated_base_number values
-- Valid pattern: B followed by exactly 5 digits e.g. B02395
SELECT 'Dirty affiliated_base_number values' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.for_hire_vehicles
WHERE affiliated_base_number IS NOT NULL
  AND affiliated_base_number NOT RLIKE '^B[0-9]{5}$'

UNION ALL

-- Null pickup location IDs — expected for pre-2017 data (lat/lon era)
SELECT 'Null pulocationid' AS check_name,
       COUNT(*) AS result
FROM group3_gp.testing.for_hire_vehicles
WHERE pulocationid IS NULL

In [0]:
%sql
-- FHV — sr_flag distribution
-- null = non-shared ride, '1' = shared ride
-- Only populated for certain years
SELECT
    CASE
        WHEN sr_flag IS NULL THEN 'Non-shared (null)'
        WHEN sr_flag = '1'   THEN 'Shared ride'
        ELSE sr_flag
    END AS sr_flag_desc,
    COUNT(*) AS trip_count
FROM group3_gp.testing.for_hire_vehicles
GROUP BY sr_flag
ORDER BY trip_count DESC

In [0]:
%sql
-- FHV — Dirty affiliated_base_number samples
-- Shows values that don't match the expected B##### pattern
SELECT
    affiliated_base_number,
    COUNT(*) AS occurrences
FROM group3_gp.testing.for_hire_vehicles
WHERE affiliated_base_number IS NOT NULL
  AND affiliated_base_number NOT RLIKE '^B[0-9]{5}$'
GROUP BY affiliated_base_number
ORDER BY occurrences DESC

In [0]:
%sql
-- FHV — Year distribution
-- Shows trip volume per year to spot coverage gaps or ingestion issues
SELECT Year, COUNT(*) AS trip_count
FROM group3_gp.testing.for_hire_vehicles
GROUP BY Year
ORDER BY Year